# GoPro 屏幕提取器
**用法：** 点击顶部菜单 **Runtime → Run all**，等待最后一个格子出现链接，点击即可使用。

In [ ]:
# 1. 安装依赖
!pip install -q flask pyngrok opencv-python-headless Pillow

In [ ]:
# 2. 核心处理逻辑
extractor_code = '''
import cv2
import numpy as np

def enhance_frame(frame):
    lab = cv2.cvtColor(frame, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    frame = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)
    blur = cv2.GaussianBlur(frame, (0, 0), 3)
    return cv2.addWeighted(frame, 1.5, blur, -0.5, 0)

def order_points(pts):
    rect = np.zeros((4, 2), dtype="float32")
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect

def detect_screen_contour(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    gray = cv2.bilateralFilter(gray, 9, 75, 75)
    edges = cv2.Canny(gray, 30, 100)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    edges = cv2.dilate(edges, kernel, iterations=2)
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours = sorted(contours, key=cv2.contourArea, reverse=True)
    frame_area = h * w
    best, best_area = None, 0
    for cnt in contours[:15]:
        area = cv2.contourArea(cnt)
        if area < frame_area * 0.05 or area > frame_area * 0.95:
            continue
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)
        if len(approx) == 4:
            pts = approx.reshape(4, 2).astype("float32")
            ordered = order_points(pts)
            width = np.linalg.norm(ordered[1] - ordered[0])
            height = np.linalg.norm(ordered[3] - ordered[0])
            if height == 0: continue
            ratio = width / height
            if 0.5 < ratio < 3.0 and area > best_area:
                best, best_area = ordered, area
    return best

def compute_output_size(pts):
    (tl, tr, br, bl) = pts
    W = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    H = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))
    return (W // 2) * 2, (H // 2) * 2

def warp_frame(frame, pts, W, H):
    dst = np.array([[0,0],[W-1,0],[W-1,H-1],[0,H-1]], dtype="float32")
    M = cv2.getPerspectiveTransform(pts, dst)
    return cv2.warpPerspective(frame, M, (W, H))

def smooth_corners(history, new_pts, alpha=0.15):
    if not history: return new_pts
    return history[-1] * (1 - alpha) + new_pts * alpha

def refine_corners_with_flow(prev_gray, curr_gray, prev_pts):
    pts_flat = prev_pts.reshape(4, 1, 2).astype("float32")
    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(
        prev_gray, curr_gray, pts_flat, None,
        winSize=(31,31), maxLevel=4,
        criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    if status is None or status.sum() < 4: return None
    return next_pts.reshape(4, 2)

def process_video(input_path, output_path, progress_callback=None):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise ValueError(f"Cannot open video: {input_path}")
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    initial_pts = None
    for _ in range(min(90, total_frames)):
        ret, frame = cap.read()
        if not ret: break
        pts = detect_screen_contour(frame)
        if pts is not None:
            initial_pts = pts
            break
    if initial_pts is None:
        cap.release()
        raise ValueError("Could not detect a screen rectangle in the video.")
    out_W, out_H = compute_output_size(initial_pts)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (out_W, out_H))
    current_pts = initial_pts.copy()
    corner_history = [current_pts]
    prev_gray = None
    frame_idx = 0
    detection_fail_count = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if prev_gray is not None:
            flow_pts = refine_corners_with_flow(prev_gray, gray, current_pts)
            if flow_pts is not None:
                detection_fail_count = 0
                current_pts = smooth_corners(corner_history, flow_pts, alpha=0.3)
                corner_history.append(current_pts)
            else:
                detection_fail_count += 1
        if prev_gray is None or detection_fail_count >= 15:
            new_pts = detect_screen_contour(frame)
            if new_pts is not None:
                current_pts = smooth_corners(corner_history, new_pts, alpha=0.25)
                corner_history.append(current_pts)
                detection_fail_count = 0
        prev_gray = gray
        try:
            warped = warp_frame(frame, current_pts, out_W, out_H)
            writer.write(enhance_frame(warped))
        except Exception:
            writer.write(np.zeros((out_H, out_W, 3), dtype=np.uint8))
        frame_idx += 1
        if progress_callback and frame_idx % 10 == 0:
            progress_callback(frame_idx, total_frames)
    cap.release()
    writer.release()
    return {"output_path": output_path, "width": out_W, "height": out_H, "fps": fps, "frames": frame_idx}
'''

with open('screen_extractor.py', 'w') as f:
    f.write(extractor_code)
print('screen_extractor.py written')

In [ ]:
# 3. Flask 应用
flask_code = '''
import os, uuid, threading
from flask import Flask, render_template_string, request, jsonify, send_file, abort
from screen_extractor import process_video

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 500 * 1024 * 1024

os.makedirs("uploads", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

jobs = {}
jobs_lock = threading.Lock()

HTML = open("index.html").read()

def _run_job(job_id, input_path, output_path):
    def progress(done, total):
        pct = int(done / total * 100) if total else 0
        with jobs_lock:
            jobs[job_id]["progress"] = pct
    with jobs_lock:
        jobs[job_id]["status"] = "processing"
    try:
        info = process_video(input_path, output_path, progress_callback=progress)
        with jobs_lock:
            jobs[job_id].update({"status": "done", "progress": 100, "info": info, "output_path": output_path})
    except Exception as e:
        with jobs_lock:
            jobs[job_id].update({"status": "error", "message": str(e)})
    finally:
        try: os.remove(input_path)
        except: pass

@app.route("/")
def index(): return render_template_string(HTML)

@app.route("/upload", methods=["POST"])
def upload():
    if "video" not in request.files: return jsonify({"error": "No file"}), 400
    f = request.files["video"]
    ext = os.path.splitext(f.filename)[1].lower()
    if ext not in (".mp4",".mov",".avi",".mkv",".m4v",".3gp"): return jsonify({"error": "Unsupported format"}), 400
    job_id = str(uuid.uuid4())
    input_path = f"uploads/{job_id}{ext}"
    output_path = f"outputs/{job_id}_extracted.mp4"
    f.save(input_path)
    with jobs_lock: jobs[job_id] = {"status": "pending", "progress": 0}
    threading.Thread(target=_run_job, args=(job_id, input_path, output_path), daemon=True).start()
    return jsonify({"job_id": job_id})

@app.route("/status/<job_id>")
def status(job_id):
    with jobs_lock: job = jobs.get(job_id)
    if job is None: abort(404)
    return jsonify(job)

@app.route("/download/<job_id>")
def download(job_id):
    with jobs_lock: job = jobs.get(job_id)
    if job is None or job.get("status") != "done": abort(404)
    path = job.get("output_path", "")
    if not os.path.exists(path): abort(404)
    return send_file(path, mimetype="video/mp4", as_attachment=True, download_name="extracted_screen.mp4")

if __name__ == "__main__":
    app.run(port=5000)
'''

with open('app.py', 'w') as f:
    f.write(flask_code)
print('app.py written')

In [ ]:
# 4. 复制 HTML 模板
import urllib.request
url = "https://raw.githubusercontent.com/lzysama/HelloWorld/claude/gopro-screen-extraction-2UQNf/templates/index.html"
urllib.request.urlretrieve(url, "index.html")

# Patch for render_template_string (no templates/ folder needed)
with open("index.html") as f:
    content = f.read()
print(f"HTML loaded: {len(content)} chars")

In [ ]:
# 5. 启动服务 + 生成公开链接
import threading, time
from pyngrok import ngrok

# Start Flask in background thread
def run_flask():
    import subprocess
    subprocess.run(["python", "app.py"])

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(3)

# Open ngrok tunnel
public_url = ngrok.connect(5000)
print("\n" + "="*50)
print(f"  在手机浏览器打开此链接：")
print(f"  {public_url}")
print("="*50)
print("\n保持此 Colab 标签页开启，链接才有效。")